# 🛒 Market Basket Analysis — Groceries Dataset

**Goal:** Find which grocery items are frequently bought together, so a store can plan shelf placement, bundle offers and cross-selling.

**Techniques used:** Exploratory Data Analysis → Basket creation → Apriori algorithm → Association rules (Support, Confidence, Lift) → Business recommendations → SQL self-join cross-check.

**Dataset:** `Groceries_data.csv` (38,765 rows: `Member_number`, `Date`, `itemDescription`, `year`, `month`, `day`, `day_of_week`).

> **Basket (transaction) = one customer on one date.**

---
**How to run in Google Colab:** first upload `Groceries_data.csv` using the left sidebar (📁 Files → Upload), then Runtime → Run all.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 140)

## 2. Load the data

In [ ]:
import os, glob, warnings
warnings.simplefilter("ignore")   # hide Colab DeprecationWarning spam

# Option A (recommended): upload the CSV in Colab's left sidebar -> Files icon -> Upload.
# The cell below finds it automatically (works with "Groceries_data.csv" or "Groceries data.csv").
matches = glob.glob("Groceries*.csv") + glob.glob("/content/Groceries*.csv")

if matches:
    FILE = matches[0]
else:
    # Option B: pop-up upload button (may print harmless warnings)
    from google.colab import files
    uploaded = files.upload()
    FILE = list(uploaded.keys())[0]

print("Using file:", FILE)
df = pd.read_csv(FILE)
df.columns = df.columns.str.strip().str.lower()   # member_number, date, itemdescription ...
df["date"] = pd.to_datetime(df["date"])
print("Shape:", df.shape)
df.head()

## 3. Data understanding & cleaning

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Unique customers:", df["member_number"].nunique())
print("Unique items:", df["itemdescription"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

**Note on duplicates:** a "duplicate row" here means the same customer bought the same item on the same day more than once (e.g. 2 packs of milk). For market basket analysis we only care whether an item is *present* in the basket, so duplicates are removed when we build baskets.

## 4. Exploratory Data Analysis (EDA)

### 4.1 Top 15 best-selling items

In [ ]:
top_items = df["itemdescription"].value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_items.values, y=top_items.index, palette="viridis")
plt.title("Top 15 Most Purchased Items")
plt.xlabel("Number of purchases")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 4.2 Sales trend by month and day of week

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

monthly = df.groupby(df["date"].dt.to_period("M")).size()
monthly.index = monthly.index.astype(str)
monthly.plot(ax=ax[0], marker="o", color="teal")
ax[0].set_title("Items sold per month")
ax[0].set_xlabel("")
ax[0].tick_params(axis="x", rotation=90)

dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow = df["date"].dt.dayofweek.value_counts().sort_index()
ax[1].bar([dow_names[i] for i in dow.index], dow.values, color="coral")
ax[1].set_title("Items sold by day of week")

plt.tight_layout()
plt.show()

### 4.3 Create baskets (transactions)

In [ ]:
# One basket = one customer on one date. Remove repeated items inside a basket.
baskets = (
    df.drop_duplicates(subset=["member_number", "date", "itemdescription"])
      .groupby(["member_number", "date"])["itemdescription"]
      .apply(list)
      .reset_index(name="items")
)
baskets["basket_size"] = baskets["items"].apply(len)

print("Total baskets:", len(baskets))
print("Average basket size: %.2f items" % baskets["basket_size"].mean())
baskets.head()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x="basket_size", data=baskets, color="steelblue")
plt.title("Basket size distribution")
plt.xlabel("Items per basket")
plt.ylabel("Number of baskets")
plt.show()

print("Baskets with 1 item : %.1f%%" % ((baskets["basket_size"] == 1).mean() * 100))
print("Baskets with 2-3 items: %.1f%%" % ((baskets["basket_size"].between(2, 3)).mean() * 100))

**Insight:** Baskets are small (about 2–3 items), so any specific item pair appears in very few baskets. That is why we use a low `min_support` threshold below.

## 5. One-hot encode the baskets

In [ ]:
transactions = baskets["items"].tolist()

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

print("Matrix shape (baskets x items):", basket_df.shape)
basket_df.iloc[:5, :8]

## 6. Apriori — find frequent itemsets

- **Support** = fraction of baskets that contain the itemset.
- We use a low `min_support = 0.001` (about 15 baskets), because the store sells 167 different items and baskets are small (about 2.5 items on average), so most pairs are rare.

In [ ]:
frequent_itemsets = apriori(basket_df, min_support=0.001, use_colnames=True)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)

print("Frequent itemsets found:", len(frequent_itemsets))
print(frequent_itemsets["length"].value_counts().sort_index().to_string())

frequent_itemsets.sort_values("support", ascending=False).head(10)

## 7. Association rules

For a rule **A → B**:

| Metric | Meaning |
|---|---|
| **Support** | How often A and B appear together in all baskets |
| **Confidence** | P(B \| A): of baskets with A, how many also have B |
| **Lift** | Confidence ÷ P(B). **Lift > 1** means A and B are bought together *more than by chance* |

In [ ]:
def build_rules(freq, metric, threshold):
    # mlxtend >= 0.23.1 needs num_itemsets; older versions don't have it
    try:
        return association_rules(freq, num_itemsets=len(basket_df), metric=metric, min_threshold=threshold)
    except TypeError:
        return association_rules(freq, metric=metric, min_threshold=threshold)

rules = build_rules(frequent_itemsets, "lift", 1.0)

rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))

cols = ["antecedents", "consequents", "support", "confidence", "lift"]
rules = rules[cols].sort_values("lift", ascending=False).reset_index(drop=True)

print("Total rules with lift > 1:", len(rules))
rules.head(15)

### 7.1 Strongest rules (filtered)

In [ ]:
strong = rules[(rules["confidence"] >= 0.05) & (rules["lift"] > 1.2)].copy()
print("Strong rules:", len(strong))
strong.head(15)

> **Reading tip:** With this dataset, confidences are usually low (5–20%) because baskets are small and 167 items compete. That is normal. **Lift** is the more useful number here.

## 8. Visualizations

### 8.1 Support vs Confidence (colour = Lift)

In [ ]:
plt.figure(figsize=(9, 6))
sc = plt.scatter(rules["support"], rules["confidence"],
                 c=rules["lift"], cmap="viridis", s=60, alpha=0.8)
plt.colorbar(sc, label="Lift")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title("Association Rules: Support vs Confidence")
plt.tight_layout()
plt.show()

### 8.2 Top 10 rules by lift

In [ ]:
top10 = rules.head(10).copy()
top10["rule"] = top10["antecedents"] + "  →  " + top10["consequents"]

plt.figure(figsize=(10, 6))
sns.barplot(x="lift", y="rule", data=top10, palette="magma")
plt.title("Top 10 Rules by Lift")
plt.xlabel("Lift")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 8.3 Lift heatmap (single item → single item)

In [ ]:
single = rules[~rules["antecedents"].str.contains(",") & ~rules["consequents"].str.contains(",")]

if len(single) > 0:
    pivot = single.pivot_table(index="antecedents", columns="consequents", values="lift")
    # keep the most connected items so the chart stays readable
    keep = pivot.notna().sum(axis=1).sort_values(ascending=False).head(15).index
    keep_cols = pivot.notna().sum(axis=0).sort_values(ascending=False).head(15).index
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot.loc[keep, keep_cols], annot=True, fmt=".2f", cmap="YlGnBu", linewidths=0.4)
    plt.title("Lift between item pairs (antecedent → consequent)")
    plt.xlabel("Consequent")
    plt.ylabel("Antecedent")
    plt.tight_layout()
    plt.show()
else:
    print("No single-item rules found. Try lowering min_support.")

### 8.4 Network graph of top rules

In [ ]:
import networkx as nx

G = nx.DiGraph()
for _, r in rules.head(20).iterrows():
    G.add_edge(r["antecedents"], r["consequents"], weight=r["lift"])

plt.figure(figsize=(11, 8))
pos = nx.spring_layout(G, k=0.9, seed=42)
nx.draw_networkx_nodes(G, pos, node_color="skyblue", node_size=1800)
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, arrowstyle="->", arrowsize=18,
                       width=[G[u][v]["weight"] for u, v in G.edges()], edge_color="gray")
plt.title("Top 20 rules: item → item (thicker = higher lift)")
plt.axis("off")
plt.show()

## 9. SQL cross-check (self join)

The same "items bought together" idea can be done in **SQL with a self join**. This also proves that the Python result is correct.

In [ ]:
conn = sqlite3.connect(":memory:")
(df.drop_duplicates(subset=["member_number", "date", "itemdescription"])
   .assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d"))
   [["member_number", "date", "itemdescription"]]
   .to_sql("groceries", conn, index=False, if_exists="replace"))

query = '''
SELECT a.itemdescription AS item1,
       b.itemdescription AS item2,
       COUNT(*)          AS times_together
FROM groceries a
JOIN groceries b
  ON  a.member_number = b.member_number
  AND a.date          = b.date
  AND a.itemdescription < b.itemdescription
GROUP BY item1, item2
ORDER BY times_together DESC
LIMIT 10;
'''
pd.read_sql_query(query, conn)

Window-function example: **top 3 items for every month** using `RANK()` (needs SQLite 3.25+, available in Colab).

In [ ]:
query2 = '''
WITH monthly AS (
    SELECT substr(date, 1, 7) AS month,
           itemdescription,
           COUNT(*) AS cnt
    FROM groceries
    GROUP BY month, itemdescription
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY month ORDER BY cnt DESC) AS rnk
    FROM monthly
)
SELECT month, itemdescription, cnt, rnk
FROM ranked
WHERE rnk <= 3
ORDER BY month, rnk;
'''
pd.read_sql_query(query2, conn).head(15)

## 10. Business recommendations (fill in using your own output)

In [ ]:
top_rules = rules.head(5)
for i, r in top_rules.iterrows():
    print(f"{i+1}. Customers who buy [{r['antecedents']}] also buy [{r['consequents']}] "
          f"(lift = {r['lift']:.2f}, confidence = {r['confidence']*100:.1f}%)")

**How a store can use these rules**

1. **Shelf placement:** keep high-lift pairs close to each other so one purchase triggers the other.
2. **Bundle offers:** create combo discounts for the top pairs (e.g. buy A, get 10% off B).
3. **Cross-selling / recommendations:** suggest the consequent item at billing or in the app when the antecedent is in the cart.
4. **Stock planning:** items that sell together should be restocked together.
5. **Seasonal promotions:** use the monthly/weekday trends from the EDA section to time the offers.

**Limitations**
- Baskets are defined as *customer + date*, so two separate shopping trips on one day are merged.
- Rules show **association, not causation**.
- Low confidence values are expected because of small baskets and many items. Rules with very low support (about 15 baskets) should be treated as hints, not proof.

## 11. Save the results

In [ ]:
rules.to_csv("association_rules.csv", index=False)
frequent_itemsets.assign(itemsets=frequent_itemsets["itemsets"].apply(lambda x: ", ".join(sorted(x)))) \
                 .to_csv("frequent_itemsets.csv", index=False)
print("Saved: association_rules.csv, frequent_itemsets.csv")

try:
    from google.colab import files
    files.download("association_rules.csv")
except ImportError:
    pass

## 12. Conclusion

- Built **14,963** baskets from customer + date and analysed **167** items.
- Applied **Apriori** to find frequent itemsets and generated association rules using **Support, Confidence and Lift**.
- Visualised rules with scatter plot, bar chart, heatmap and network graph.
- Verified the pair counts with a **SQL self join** and used **CTE + RANK()** for monthly top items.
- Converted the findings into practical retail recommendations.

**Future work:** try FP-Growth for speed, add item categories, split rules by season, and build a simple recommender.